# Bronze Ingest: NOAA GSOD Daily Weather

Downloads a year of NOAA Global Surface Summary of the Day (GSOD)
observations for one weather station and persists it as a managed Spark
table, unmodified from source (including NOAA's missing-value sentinels
like `9999.9` / `999.9`). This is the `bronze` layer — cleaning those
sentinels into real nulls happens in `data-transformation/`.

Source: AWS Open Data (`noaa-gsod-pds`), no auth required. Station IDs are
listed at https://www1.ncdc.noaa.gov/pub/data/noaa/isd-history.csv (this
one is Seattle Boeing Field, WA).

Re-runnable: uses `CREATE DATABASE IF NOT EXISTS` and `mode("overwrite")`,
so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Configuration

Change `STATION` / `YEAR` to pull a different station or year without
touching the logic below.

In [ ]:
DATABASE_NAME = "noaa_weather"
TABLE_NAME = "daily_observations"
STATION = "72793524234"  # Seattle Boeing Field, WA
YEAR = "2025"
SOURCE_URL = f"https://noaa-gsod-pds.s3.amazonaws.com/{YEAR}/{STATION}.csv"
LOCAL_FILE = f"/tmp/noaa_gsod_{STATION}_{YEAR}.csv"

## Download source file

In [ ]:
import urllib.request

urllib.request.urlretrieve(SOURCE_URL, LOCAL_FILE)

## Load dataset

In [ ]:
df = spark.read.csv(LOCAL_FILE, header=True, inferSchema=True)

## Create schema/database

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
COMMENT 'NOAA GSOD daily weather observations used for analytics demonstrations'
""")

## Persist as a managed Spark table

In [ ]:
df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TABLE_NAME}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_days FROM {DATABASE_NAME}.{TABLE_NAME}").show()
spark.sql(f"DESCRIBE {DATABASE_NAME}.{TABLE_NAME}").show(truncate=False)